# 02 · Silver
Limpa, tipa e valida. O que não passa vai para as tabelas `quarentena_*`, com o motivo da rejeição.

In [ ]:
# Pré-requisito: a pasta src/medallion do repositório foi enviada para
# Files/medallion_src/medallion no Lakehouse padrão (veja o README, passo 2).
import sys
sys.path.append("/lakehouse/default/Files/medallion_src")

import pandas as pd
from medallion import bronze, silver, gold, quality, fabric, rules
from medallion.generate import generate_raw

RAW = f"{fabric.LAKEHOUSE_FILES}/raw"
HOJE = "2025-08-01"  # data de referência para detectar datas futuras

In [ ]:
b = {t: fabric.read_delta(spark, f"bronze_{t}") for t in rules.TABELAS_RAW}

escolas, rej_escolas = silver.clean_escolas(b["escolas"])
turmas, rej_turmas = silver.clean_turmas(b["turmas"], escolas)
alunos, rej_alunos = silver.clean_alunos(b["alunos"], turmas)
freq, rej_freq = silver.clean_frequencia(b["frequencia"], alunos, HOJE)

In [ ]:
limpas = {"escolas": escolas, "turmas": turmas, "alunos": alunos, "frequencia": freq}
rejeitadas = {"escolas": rej_escolas, "turmas": rej_turmas, "alunos": rej_alunos, "frequencia": rej_freq}

for nome, df in limpas.items():
    fabric.save_delta(spark, df, f"silver_{nome}")
for nome, df in rejeitadas.items():
    fabric.save_delta(spark, df, f"quarentena_{nome}")
    print(f"{nome}: {len(limpas[nome])} limpas, {len(df)} em quarentena")

In [ ]:
display(rej_freq["motivo_rejeicao"].value_counts().rename_axis("motivo").reset_index(name="linhas"))